In [1]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt



def charger_grille(chemin_image, n_cols, n_rows, taille_pixel=28):
    """
    Découpe une image grille en chiffres individuels 28×28.
    Retourne un tableau numpy (N, 784).
    """
    img = Image.open(chemin_image).convert('L')  # niveaux de gris
    img_array = np.array(img)

    H, W = img_array.shape
    h_chiffre = H // n_rows
    w_chiffre = W // n_cols

    images = []
    for row in range(n_rows):
        for col in range(n_cols):
            y0 = row * h_chiffre
            y1 = y0 + h_chiffre
            x0 = col * w_chiffre
            x1 = x0 + w_chiffre

            chiffre = img_array[y0:y1, x0:x1]
            # Redimensionner à 28×28 si nécessaire
            chiffre_pil = Image.fromarray(chiffre).resize((28, 28))
            chiffre_vec = np.array(chiffre_pil).flatten()  # vecteur ℝ^784
            images.append(chiffre_vec)

    return np.array(images)

In [2]:
from tensorflow.keras.datasets import mnist  # ou : sklearn, torchvision

(X_train, y_train), (X_test, y_test) = mnist.load_data()

# X_train : (60000, 28, 28) → on vectorise en (60000, 784)
X_train = X_train.reshape(60000, 784)
X_test = X_test.reshape(10000, 784)

# ── 3. Normalisation ─────────────────────────────────────────────────────────
# Pixels entre 0 et 255 → on ramène entre 0 et 1
X_train = X_train / 255.0
X_test = X_test / 255.0

print(f"X_train : {X_train.shape}, valeurs : [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"X_test  : {X_test.shape}")
print(f"Classes présentes : {np.unique(y_train)}")

X_train : (60000, 784), valeurs : [0.00, 1.00]
X_test  : (10000, 784)
Classes présentes : [0 1 2 3 4 5 6 7 8 9]


In [3]:
def afficher_grille(X, y, n=10):
    fig, axes = plt.subplots(1, n, figsize=(15, 2))
    for i, ax in enumerate(axes):
        ax.imshow(X[i].reshape(28, 28), cmap='gray')
        ax.set_title(f"y={y[i]}")
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [4]:
import numpy as np

# ── Initialisation ─────────────────────────────
n_features = 784
n_classes = 10

W = np.random.randn(n_classes, n_features) * 0.01
b = np.zeros((n_classes, 1))

# ── Softmax ───────────────────────────────────
def softmax(z):
    z = z - np.max(z, axis=0, keepdims=True)  # stabilité numérique
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

# ── One-hot ───────────────────────────────────
def one_hot(y, num_classes=10):
    oh = np.zeros((num_classes, y.size))
    oh[y, np.arange(y.size)] = 1
    return oh

# ── Entraînement ─────────────────────────────
def train(X, y, lr=0.1, epochs=10):
    global W, b

    X = X.T  # shape (784, N)
    y_onehot = one_hot(y)

    for epoch in range(epochs):

        # Forward
        z = W @ X + b
        y_hat = softmax(z)

        # Loss
        loss = -np.mean(np.sum(y_onehot * np.log(y_hat + 1e-9), axis=0))

        # Gradient
        dz = y_hat - y_onehot
        dW = (dz @ X.T) / X.shape[1]
        db = np.mean(dz, axis=1, keepdims=True)

        # Update
        W -= lr * dW
        b -= lr * db

        print(f"Epoch {epoch+1}, Loss = {loss:.4f}")

# ── Prédiction ────────────────────────────────
def predict(X):
    X = X.T
    z = W @ X + b
    y_hat = softmax(z)
    return np.argmax(y_hat, axis=0)

In [5]:
train(X_train, y_train, lr=0.1, epochs=20)

y_pred = predict(X_test)

accuracy = np.mean(y_pred == y_test)
print("Accuracy :", accuracy)

Epoch 1, Loss = 2.3024
Epoch 2, Loss = 2.1964
Epoch 3, Loss = 2.0999
Epoch 4, Loss = 2.0110
Epoch 5, Loss = 1.9287
Epoch 6, Loss = 1.8524
Epoch 7, Loss = 1.7817
Epoch 8, Loss = 1.7161
Epoch 9, Loss = 1.6553
Epoch 10, Loss = 1.5989
Epoch 11, Loss = 1.5467
Epoch 12, Loss = 1.4982
Epoch 13, Loss = 1.4531
Epoch 14, Loss = 1.4112
Epoch 15, Loss = 1.3722
Epoch 16, Loss = 1.3358
Epoch 17, Loss = 1.3019
Epoch 18, Loss = 1.2702
Epoch 19, Loss = 1.2405
Epoch 20, Loss = 1.2126
Accuracy : 0.8174
